# Notebook 4: NLP — LLM Erklärungen & Prompt Engineering
**Skin Lesion Risk Advisor**

In diesem Notebook entwickeln wir den NLP-Block:
1. Verschiedene Prompt-Strategien designen
2. Prompts vergleichen (qualitative Evaluation)
3. Beste Prompt-Strategie auswählen
4. Integration mit CV + ML Output testen

**Ziel:** Das LLM soll aus CV-Diagnose + ML-Risikoklasse + Patientendaten eine **verständliche Erklärung** auf Deutsch generieren.

## 0. Setup

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd
from pathlib import Path
import json

load_dotenv('../.env')
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

MODEL = 'gpt-4o-mini'  # Kostengünstig und gut genug
print(f'LLM: {MODEL}')
print('Setup OK')

LLM: gpt-4o-mini
Setup OK


## 1. Test-Szenarien definieren

Wir testen 3 verschiedene Patientenszenarien mit bekanntem Output von CV + ML.

In [2]:
# Simulierte Outputs aus CV + ML Pipeline
test_cases = [
    {
        'patient': {'age': 62, 'sex': 'male', 'localization': 'back'},
        'cv_prediction': 'mel',
        'cv_confidence': 0.78,
        'risk_level': 'high',
        'label': 'Melanoma (hohes Risiko)'
    },
    {
        'patient': {'age': 34, 'sex': 'female', 'localization': 'lower extremity'},
        'cv_prediction': 'nv',
        'cv_confidence': 0.91,
        'risk_level': 'low',
        'label': 'Gutartiger Leberfleck (niedriges Risiko)'
    },
    {
        'patient': {'age': 55, 'sex': 'female', 'localization': 'face'},
        'cv_prediction': 'akiec',
        'cv_confidence': 0.65,
        'risk_level': 'medium',
        'label': 'Aktinische Keratose (mittleres Risiko)'
    }
]

dx_names = {
    'nv':    'Melanocytic Nevus (gutartiger Leberfleck)',
    'mel':   'Melanoma (bösartig)',
    'bkl':   'Benigne Keratose',
    'bcc':   'Basalzellkarzinom',
    'akiec': 'Aktinische Keratose (Vorstufe)',
    'vasc':  'Vaskuläre Läsion',
    'df':    'Dermatofibrom'
}

print(f'{len(test_cases)} Testszenarien definiert.')

3 Testszenarien definiert.


## 2. Prompt-Strategie A: Einfacher Prompt (Baseline)

In [3]:
def prompt_simple(case):
    dx = dx_names[case['cv_prediction']]
    return f"""Eine KI hat eine Hautläsion als '{dx}' klassifiziert.
Das Risikolevel ist '{case['risk_level']}'.
Patient: {case['patient']['age']} Jahre, {case['patient']['sex']}, Körperstelle: {case['patient']['localization']}.

Erkläre das Ergebnis in 2-3 Sätzen auf Deutsch für den Patienten."""

def call_llm(prompt, temperature=0.3):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'Du bist ein hilfreicher medizinischer Assistent der Patienten informiert. Du bist kein Arzt und gibst keine Diagnosen.'},
            {'role': 'user', 'content': prompt}
        ],
        temperature=temperature,
        max_tokens=300
    )
    return response.choices[0].message.content

print('=== Strategie A: Einfacher Prompt ===')
for case in test_cases:
    print(f'\n--- {case["label"]} ---')
    prompt = prompt_simple(case)
    response = call_llm(prompt)
    print(response)

=== Strategie A: Einfacher Prompt ===

--- Melanoma (hohes Risiko) ---
Die KI hat Ihre Hautläsion als bösartiges Melanom eingestuft, was bedeutet, dass es sich um eine Form von Hautkrebs handelt, die potenziell gefährlich sein kann. Das hohe Risikolevel weist darauf hin, dass es wichtig ist, schnell zu handeln und einen Facharzt aufzusuchen, um weitere Untersuchungen und mögliche Behandlungen zu besprechen. Es ist entscheidend, die Situation ernst zu nehmen und die nächsten Schritte zu planen.

--- Gutartiger Leberfleck (niedriges Risiko) ---
Das Ergebnis zeigt, dass die Hautläsion als gutartiger Leberfleck eingestuft wurde, was bedeutet, dass sie in der Regel keine gesundheitlichen Bedenken verursacht. Das niedrige Risikolevel deutet darauf hin, dass keine Anzeichen für eine bösartige Veränderung vorliegen. Es ist jedoch wichtig, die Haut regelmäßig zu beobachten und bei Veränderungen einen Arzt aufzusuchen.

--- Aktinische Keratose (mittleres Risiko) ---
Aktinische Keratose ist eine 

## 3. Prompt-Strategie B: Strukturierter Prompt mit Kontext

In [4]:
def prompt_structured(case):
    dx = dx_names[case['cv_prediction']]
    conf_pct = int(case['cv_confidence'] * 100)
    risk_emoji = {'high': '🔴', 'medium': '🟡', 'low': '🟢'}[case['risk_level']]
    
    return f"""Du bist ein KI-Assistent in einer Hautkrebs-Früherkennungs-App.

KI-Analyseergebnis:
- Bildanalyse: {dx} (Konfidenz: {conf_pct}%)
- Risikostufe: {risk_emoji} {case['risk_level'].upper()}
- Patient: {case['patient']['age']} Jahre, {case['patient']['sex']}
- Betroffene Stelle: {case['patient']['localization']}

Schreibe eine strukturierte Erklärung auf Deutsch mit:
1. Was die KI gefunden hat (1 Satz)
2. Was das bedeutet (1-2 Sätze, verständlich)
3. Empfehlung: Was soll der Patient jetzt tun? (1 Satz)
4. Wichtiger Hinweis: Dies ist keine ärztliche Diagnose.

Schreib direkt, klar und ohne Fachjargon."""

print('=== Strategie B: Strukturierter Prompt ===')
for case in test_cases:
    print(f'\n--- {case["label"]} ---')
    prompt = prompt_structured(case)
    response = call_llm(prompt)
    print(response)

=== Strategie B: Strukturierter Prompt ===

--- Melanoma (hohes Risiko) ---
1. Die KI hat Anzeichen für ein bösartiges Melanom mit einer hohen Wahrscheinlichkeit von 78% festgestellt.  
2. Das bedeutet, dass es sich um eine potenziell gefährliche Hautveränderung handeln könnte, die behandelt werden sollte.  
3. Wir empfehlen Ihnen dringend, einen Hautarzt aufzusuchen, um die Stelle untersuchen und gegebenenfalls biopsieren zu lassen.  
4. Wichtiger Hinweis: Dies ist keine ärztliche Diagnose.

--- Gutartiger Leberfleck (niedriges Risiko) ---
1. Die KI hat einen gutartigen Leberfleck (Melanocytic Nevus) mit einer hohen Sicherheit von 91% identifiziert.  
2. Das bedeutet, dass der Leberfleck wahrscheinlich unbedenklich ist und kein erhöhtes Risiko für Hautkrebs darstellt.  
3. Es wird empfohlen, den Leberfleck regelmäßig zu beobachten und bei Veränderungen einen Hautarzt aufzusuchen.  
4. Wichtiger Hinweis: Dies ist keine ärztliche Diagnose.

--- Aktinische Keratose (mittleres Risiko) ---

## 4. Prompt-Strategie C: Risiko-adaptiver Prompt

In [5]:
def prompt_adaptive(case):
    dx = dx_names[case['cv_prediction']]
    risk = case['risk_level']
    
    tone_map = {
        'high': 'ernst und dringend, aber beruhigend',
        'medium': 'besorgt aber nicht alarmistisch',
        'low': 'beruhigend und positiv'
    }
    action_map = {
        'high': 'Bitte suchen Sie umgehend einen Dermatologen auf.',
        'medium': 'Ein Besuch beim Dermatologen innerhalb der nächsten Wochen wird empfohlen.',
        'low': 'Beobachten Sie die Stelle und konsultieren Sie bei Veränderungen einen Arzt.'
    }
    
    return f"""Kontext: Hautläsions-Analyse App.

Analyseergebnis:
- Typ: {dx}
- Risiko: {risk}
- Alter: {case['patient']['age']} | Geschlecht: {case['patient']['sex']} | Stelle: {case['patient']['localization']}

Tonalität: {tone_map[risk]}
Standard-Empfehlung: {action_map[risk]}

Schreibe eine Erklärung auf Deutsch (max. 100 Wörter) die:
- Den Befund klar erklärt
- Die Empfehlung einbezieht
- Mit einem Disclaimer endet: 'Hinweis: Diese Analyse ersetzt keine ärztliche Untersuchung.'"""

print('=== Strategie C: Risiko-adaptiver Prompt ===')
for case in test_cases:
    print(f'\n--- {case["label"]} ---')
    prompt = prompt_adaptive(case)
    response = call_llm(prompt)
    print(response)

=== Strategie C: Risiko-adaptiver Prompt ===

--- Melanoma (hohes Risiko) ---
Ihr Analyseergebnis zeigt, dass die Hautläsion als bösartiges Melanom eingestuft wurde, was ein hohes Risiko für gesundheitliche Komplikationen bedeutet. Angesichts Ihres Alters und Geschlechts ist es besonders wichtig, schnell zu handeln. Wir empfehlen Ihnen dringend, umgehend einen Dermatologen aufzusuchen, um eine umfassende Untersuchung und gegebenenfalls eine Behandlung einzuleiten. Ihre Gesundheit hat oberste Priorität. 

Hinweis: Diese Analyse ersetzt keine ärztliche Untersuchung.

--- Gutartiger Leberfleck (niedriges Risiko) ---
Ihr Analyseergebnis zeigt, dass es sich um einen Melanozytären Nevus handelt, einen gutartigen Leberfleck, der ein niedriges Risiko für gesundheitliche Probleme aufweist. Dies ist in der Regel unbedenklich. Es wird empfohlen, die Stelle weiterhin zu beobachten und bei Veränderungen, wie z.B. Größe, Farbe oder Form, einen Arzt aufzusuchen. So stellen Sie sicher, dass alles in O

## 5. Qualitativer Vergleich der Strategien

In [6]:
# Bewertungskriterien (manuell bewertet, 1-5)
comparison = pd.DataFrame({
    'Kriterium': [
        'Verständlichkeit',
        'Vollständigkeit',
        'Risiko-Anpassung',
        'Sicherheits-Disclaimer',
        'Länge/Präzision'
    ],
    'Strategie A (Einfach)': [3, 2, 1, 2, 3],
    'Strategie B (Strukturiert)': [4, 5, 3, 5, 4],
    'Strategie C (Adaptiv)': [5, 4, 5, 5, 5]
})

print(comparison.to_string(index=False))
print()
print('Durchschnitt:')
for col in ['Strategie A (Einfach)', 'Strategie B (Strukturiert)', 'Strategie C (Adaptiv)']:
    print(f'  {col}: {comparison[col].mean():.1f}/5')

print('\n→ Strategie C wird für die App verwendet.')

             Kriterium  Strategie A (Einfach)  Strategie B (Strukturiert)  Strategie C (Adaptiv)
      Verständlichkeit                      3                           4                      5
       Vollständigkeit                      2                           5                      4
      Risiko-Anpassung                      1                           3                      5
Sicherheits-Disclaimer                      2                           5                      5
       Länge/Präzision                      3                           4                      5

Durchschnitt:
  Strategie A (Einfach): 2.2/5
  Strategie B (Strukturiert): 4.2/5
  Strategie C (Adaptiv): 4.8/5

→ Strategie C wird für die App verwendet.


## 6. Finale Prompt-Funktion für die App

In [7]:
def generate_explanation(cv_prediction, cv_confidence, risk_level, age, sex, localization):
    """
    Finale Prompt-Funktion (Strategie C) — wird in der App verwendet.
    """
    dx_names_map = {
        'nv':    'Melanocytic Nevus (gutartiger Leberfleck)',
        'mel':   'Melanoma',
        'bkl':   'Benigne Keratose',
        'bcc':   'Basalzellkarzinom',
        'akiec': 'Aktinische Keratose',
        'vasc':  'Vaskuläre Läsion',
        'df':    'Dermatofibrom'
    }
    action_map = {
        'high':   'Bitte suchen Sie umgehend einen Dermatologen auf.',
        'medium': 'Ein Besuch beim Dermatologen innerhalb der nächsten Wochen wird empfohlen.',
        'low':    'Beobachten Sie die Stelle und konsultieren Sie bei Veränderungen einen Arzt.'
    }
    tone_map = {
        'high':   'ernst und dringend, aber beruhigend',
        'medium': 'besorgt aber nicht alarmistisch',
        'low':    'beruhigend und positiv'
    }

    prompt = f"""Kontext: Hautläsions-Analyse App.

Analyseergebnis:
- Typ: {dx_names_map.get(cv_prediction, cv_prediction)} (Konfidenz: {int(cv_confidence*100)}%)
- Risiko: {risk_level.upper()}
- Alter: {age} | Geschlecht: {sex} | Stelle: {localization}

Tonalität: {tone_map[risk_level]}
Empfehlung: {action_map[risk_level]}

Schreibe eine Erklärung auf Deutsch (max. 100 Wörter) die:
- Den Befund klar erklärt
- Die Empfehlung einbezieht
- Mit diesem Satz endet: 'Hinweis: Diese Analyse ersetzt keine ärztliche Untersuchung.'"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'Du bist ein KI-Assistent für Hautgesundheit. Du gibst keine Diagnosen, nur Informationen und Empfehlungen.'},
            {'role': 'user', 'content': prompt}
        ],
        temperature=0.3,
        max_tokens=300
    )
    return response.choices[0].message.content

# Test
result = generate_explanation(
    cv_prediction='mel',
    cv_confidence=0.78,
    risk_level='high',
    age=62,
    sex='male',
    localization='back'
)
print(result)

Ihr Analyseergebnis zeigt, dass eine Hautläsion auf Ihrem Rücken mit einer Wahrscheinlichkeit von 78% als Melanom eingestuft wurde. Dies bedeutet, dass ein hohes Risiko für Hautkrebs besteht. Angesichts Ihres Alters und Geschlechts ist es besonders wichtig, diese Ergebnisse ernst zu nehmen. Wir empfehlen Ihnen dringend, umgehend einen Dermatologen aufzusuchen, um eine gründliche Untersuchung und gegebenenfalls weitere Schritte einzuleiten. Ihre Gesundheit hat oberste Priorität. Hinweis: Diese Analyse ersetzt keine ärztliche Untersuchung.


## 7. Zusammenfassung

**Ergebnisse:**
- 3 Prompt-Strategien verglichen: Einfach / Strukturiert / Adaptiv
- **Strategie C (Adaptiv)** gewinnt: passt Ton und Empfehlung ans Risikolevel an
- LLM integriert CV-Output (Diagnosetyp, Konfidenz) + ML-Output (risk_level) + Patientendaten
- Disclaimer ist immer enthalten → wichtig für medizinische Apps

**Integration mit anderen Blöcken:**
- Input: CV-Vorhersage + Konfidenz (aus Notebook 03)
- Input: risk_level (aus Notebook 02)
- Output: Verständliche Erklärung für den Patienten

**Nächster Schritt:** Notebook 05 — Vollständige End-to-End Evaluation